# Method Comparison Notebook

Compare UAV localization methods on the same dataset split and summarize the metrics used in Table 5 of `paper_draft/main_0406.tex`:

- Mean (m)
- Med. (m)
- RMSE (m)
- R@10m (%)
- R@50m (%)
- Time (s)

Initial supported methods:

- `main` — LocalizationUAV MFCA pipeline
- `sift` — standalone classical baseline
- `orb` — standalone classical baseline


In [22]:
import os
import sys
import time
import math
from pathlib import Path
import pathlib
pathlib.PosixPath = pathlib.WindowsPath

import numpy as np
import pandas as pd
import torch
from PIL import Image

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
PAPER6_ROOT = REPO_ROOT.parent
if str(PAPER6_ROOT) not in sys.path:
    sys.path.insert(0, str(PAPER6_ROOT))

DATA_ROOT = Path(r'D:/bk_study_stuff/paper6/UAV-VisLoc')
MODEL_PATH = REPO_ROOT / 'best_model.pth'
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'method_comparison'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METHODS = ['main', 'sift', 'orb']

SITES = ['01','02','03']

# Option 1: lấy tối đa N ảnh đầu tiên cho mỗi site. Đặt None để lấy toàn bộ.
MAX_IMAGES_PER_SITE = 3

# Option 2: chọn ảnh cụ thể theo từng site. Nếu site có trong dict này thì ưu tiên list này thay cho MAX_IMAGES_PER_SITE.
# Đặt SELECTED_IMAGES = {} hoặc None nếu muốn dùng MAX_IMAGES_PER_SITE cho mọi site.
SELECTED_IMAGES = {
    '01': ['01_0514.JPG','01_0564.JPG','01_0742.JPG'],
    '02': ['02_0024.JPG', '02_0130.JPG','02_0226.JPG'],
    '03': ['03_0008.JPG','03_0126.JPG','01_0192.JPG',],
    # '04': ['04_0080.JPG'],
    # '05': ['05_0052.JPG'],
    # '06': ['06_0136.JPG'],
    # '07': ['07_0010.JPG'],
    # '08': ['08_0311.JPG'],
    # '10': ['10_0065.JPG'],
    # '11': ['11_0114.JPG'], 

}

TOP_K = 5
MAIN_TOP_N = 100
MAIN_SCORE_THRESHOLD = 0.5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Repo root :', REPO_ROOT)
print('Data root :', DATA_ROOT)
print('Model     :', MODEL_PATH)
print('Output dir:', OUTPUT_DIR)
print('Device    :', DEVICE)

Repo root : D:\bk_study_stuff\paper6\LocalizationUAV
Data root : D:\bk_study_stuff\paper6\UAV-VisLoc
Model     : D:\bk_study_stuff\paper6\LocalizationUAV\best_model.pth
Output dir: D:\bk_study_stuff\paper6\LocalizationUAV\outputs\method_comparison
Device    : cuda


In [23]:
from localization import load_model, process_uav, SatelliteDatabase, query_uav
from localization.database.builder import extract_patch_descriptors
from localization.io.bounds import load_satellite_bounds, pixel_to_latlon
from localization.io.dataset import VisLocFlight, load_flight_metadata

import SIFT_ORB.eval_sift_orb_v2 as classical_baselines

classical_baselines.METHOD_CONFIG['orb']['nfeatures'] = 5000
classical_baselines.METHOD_CONFIG['orb']['dist_thresh'] = 50

# Bạn muốn nới lỏng SIFT ratio?
classical_baselines.METHOD_CONFIG['sift']['n_features'] = 3000
classical_baselines.METHOD_CONFIG['sift']['lowe_ratio'] = 0.75


In [24]:
def build_eval_rows_dataframe(rows):
    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame(columns=[
            'method', 'site', 'filename', 'gt_lat', 'gt_lon',
            'pred_lat', 'pred_lon', 'error_m', 'matched', 'time_s'
        ])
    return df


def summarize_metrics(results_df):
    summaries = []
    for method, group in results_df.groupby('method', sort=False):
        matched = group[group['matched'] == True].copy()
        errors = matched['error_m'].astype(float).to_numpy() if not matched.empty else np.array([], dtype=float)
        times = group['time_s'].astype(float).to_numpy() if 'time_s' in group else np.array([], dtype=float)

        summaries.append({
            'Method': method,
            'Mean (m)': float(np.mean(errors)) if len(errors) else np.nan,
            'Med. (m)': float(np.median(errors)) if len(errors) else np.nan,
            'RMSE (m)': float(np.sqrt(np.mean(errors ** 2))) if len(errors) else np.nan,
            'R@10m (%)': float(np.mean(errors <= 10) * 100.0) if len(errors) else np.nan,
            'R@50m (%)': float(np.mean(errors <= 50) * 100.0) if len(errors) else np.nan,
            'Time (s)': float(np.mean(times)) if len(times) else np.nan,
            'Matched / Total': f"{int(group['matched'].sum())}/{len(group)}",
        })
    summary_df = pd.DataFrame(summaries)
    if not summary_df.empty:
        summary_df = summary_df[['Method', 'Mean (m)', 'Med. (m)', 'RMSE (m)', 'R@10m (%)', 'R@50m (%)', 'Time (s)', 'Matched / Total']]
    return summary_df


def select_flight_images(flight, metadata_df, image_names=None, max_images_per_site=None):
    rows = metadata_df.copy()
    rows['filename'] = rows['filename'].astype(str).str.strip()
    rows['lat_num'] = pd.to_numeric(rows['lat'], errors='coerce')
    rows['lon_num'] = pd.to_numeric(rows['lon'], errors='coerce')
    rows = rows[rows['lat_num'].notna() & rows['lon_num'].notna()].copy()
    rows = rows.drop(columns=['lat_num', 'lon_num'])
    if image_names is not None:
        wanted = set(image_names)
        rows = rows[rows['filename'].isin(wanted)]
    if max_images_per_site is not None:
        rows = rows.head(int(max_images_per_site))
    selected = []
    for _, row in rows.iterrows():
        image_path = flight.drone_image_path(str(row['filename']).strip())
        if image_path.exists():
            selected.append((image_path, row))
    return selected

In [25]:
import gc

_main_model = None


def get_main_model():
    global _main_model
    if _main_model is None:
        _main_model = load_model(
            model_path=str(MODEL_PATH),
            device=DEVICE,
            num_classes=2,
            pretrained=False,
        ).to(DEVICE).eval()
    return _main_model


def load_main_database(site):
    db_path = REPO_ROOT / 'outputs' / site / f'satellite{site}_kdtree.npz'
    if not db_path.exists():
        raise FileNotFoundError(
            f'Missing KD-tree database for site {site}: {db_path}. Build it first with notebook 01.'
        )
    return SatelliteDatabase.load(str(db_path))


def select_flight_images(flight, metadata_df, image_names=None, max_images_per_site=None):
    rows = metadata_df.copy()
    rows['filename'] = rows['filename'].astype(str).str.strip()
    rows['lat_num'] = pd.to_numeric(rows['lat'], errors='coerce')
    rows['lon_num'] = pd.to_numeric(rows['lon'], errors='coerce')
    rows = rows[rows['lat_num'].notna() & rows['lon_num'].notna()].copy()
    rows = rows.drop(columns=['lat_num', 'lon_num'])
    if image_names is not None:
        wanted = set(image_names)
        rows = rows[rows['filename'].isin(wanted)]
    if max_images_per_site is not None:
        rows = rows.head(int(max_images_per_site))
    selected = []
    for _, row in rows.iterrows():
        image_path = flight.drone_image_path(str(row['filename']).strip())
        if image_path.exists():
            selected.append((image_path, row))
    return selected


def run_main_method_for_site(site, image_names=None, max_images_per_site=None):
    model = get_main_model()
    db = load_main_database(site)
    flight = VisLocFlight(flight_id=site, root=DATA_ROOT)
    metadata_df = load_flight_metadata(flight.metadata_csv)
    selected = select_flight_images(flight, metadata_df, image_names=image_names, max_images_per_site=max_images_per_site)

    bounds = load_satellite_bounds(
        satellite_filename=os.path.basename(str(flight.satellite_tif)),
        csv_path=str(flight.bounds_csv),
    )
    with Image.open(flight.satellite_tif) as sat:
        sat_w, sat_h = sat.size

    rows = []
    for image_path, row in selected:
        gt_lat = float(row['lat'])
        gt_lon = float(row['lon'])
        t0 = time.time()
        try:
            _, img_uav_500, _meta = process_uav(
                img_path=str(image_path),
                csv_path=str(flight.metadata_csv),
            )
            uav_descriptors, _ = extract_patch_descriptors(
                patch_image=img_uav_500,
                model=model,
                device=DEVICE,
                score_threshold=MAIN_SCORE_THRESHOLD,
            )
            result = query_uav(uav_descriptors, db, k=TOP_K, top_n=MAIN_TOP_N) if uav_descriptors.shape[0] > 0 else None
            elapsed = time.time() - t0

            if result is None or bounds is None:
                pred_lat = None
                pred_lon = None
                error_m = float('inf')
                matched = False
                pred_x = None
                pred_y = None
                vote_count = 0
            else:
                pred_x, pred_y = float(result.pixel_xy[0]), float(result.pixel_xy[1])
                pred_lat, pred_lon = pixel_to_latlon(pred_x, pred_y, bounds, sat_w, sat_h)
                error_m = float(classical_baselines.haversine_m(gt_lat, gt_lon, pred_lat, pred_lon))
                matched = True
                vote_count = int(result.vote_count)

            rows.append({
                'method': 'main',
                'site': site,
                'filename': image_path.name,
                'gt_lat': gt_lat,
                'gt_lon': gt_lon,
                'pred_lat': pred_lat,
                'pred_lon': pred_lon,
                'pred_x': pred_x,
                'pred_y': pred_y,
                'vote_count': vote_count,
                'error_m': error_m,
                'matched': matched,
                'time_s': elapsed,
            })
        except Exception as exc:
            rows.append({
                'method': 'main',
                'site': site,
                'filename': image_path.name,
                'gt_lat': gt_lat,
                'gt_lon': gt_lon,
                'pred_lat': None,
                'pred_lon': None,
                'pred_x': None,
                'pred_y': None,
                'vote_count': 0,
                'error_m': float('inf'),
                'matched': False,
                'time_s': time.time() - t0,
                'error_note': str(exc),
            })
    del db
    gc.collect()
    return rows


def run_classical_method_for_site(site, method, image_names=None, max_images_per_site=None):
    flight = VisLocFlight(flight_id=site, root=DATA_ROOT)
    metadata_df = load_flight_metadata(flight.metadata_csv)
    selected = select_flight_images(flight, metadata_df, image_names=image_names, max_images_per_site=max_images_per_site)

    normalized = []
    for image_path, _row in selected:
        results = classical_baselines.evaluate(
            data_root=str(DATA_ROOT),
            site=site,
            method=method,
            img_name=image_path.name,
            output_csv=None,
        )
        for r in results:
            normalized.append({
                'method': r['method'],
                'site': r['site'],
                'filename': r['filename'],
                'gt_lat': r['gt_lat'],
                'gt_lon': r['gt_lon'],
                'pred_lat': r['pred_lat'],
                'pred_lon': r['pred_lon'],
                'pred_x': r.get('pred_x'),
                'pred_y': r.get('pred_y'),
                'inliers': r.get('inliers', 0),
                'error_m': r['error_m'],
                'matched': r['matched'],
                'time_s': r['time_s'],
            })
    return normalized

In [26]:
METHOD_RUNNERS = {
    'main': run_main_method_for_site,
    'sift': lambda site, image_names=None, max_images_per_site=None: run_classical_method_for_site(site, 'sift', image_names=image_names, max_images_per_site=max_images_per_site),
    'orb': lambda site, image_names=None, max_images_per_site=None: run_classical_method_for_site(site, 'orb', image_names=image_names, max_images_per_site=max_images_per_site),
}


def resolve_site_selection(site):
    selected_map = SELECTED_IMAGES or {}
    if site in selected_map and selected_map[site]:
        return list(selected_map[site]), None
    return None, MAX_IMAGES_PER_SITE


all_rows = []
for method in METHODS:
    for site in SITES:
        image_names, max_imgs = resolve_site_selection(site)
        sel_desc = image_names if image_names is not None else f'first {max_imgs}'
        print(f'Running {method} on site {site} (images: {sel_desc}) ...')
        rows = METHOD_RUNNERS[method](site, image_names=image_names, max_images_per_site=max_imgs)
        all_rows.extend(rows)

results_df = build_eval_rows_dataframe(all_rows)
results_df

Running main on site 01 (images: ['01_0514.JPG', '01_0564.JPG', '01_0742.JPG']) ...
Running main on site 02 (images: ['02_0024.JPG', '02_0130.JPG', '02_0226.JPG']) ...
Running main on site 03 (images: ['03_0008.JPG', '03_0126.JPG', '01_0192.JPG']) ...
Running sift on site 01 (images: ['01_0514.JPG', '01_0564.JPG', '01_0742.JPG']) ...
Bounds: {'LT_lat': 29.774065, 'LT_lon': 115.970635, 'RB_lat': 29.702283, 'RB_lon': 115.996851}
[SIFT] Reference status for site 01: memory
  Loaded reference cache: memory
  Offline time: 0.0s

[SIFT] Online phase: evaluating 1 UAV queries ...
  [1/1] 01_0514.JPG → 3354.1 m, px=(206.9,1575.2) (inliers=29, t=2.3s)

──────────────────────────────────────────────────
[SIFT] Site 01 — 1/1 matched
  Mean   : 3354.14 m
  Median : 3354.14 m
  RMSE   : 3354.14 m
  R@10m  : 0.0%
  R@50m  : 0.0%
  Online : 2.3s
──────────────────────────────────────────────────
Bounds: {'LT_lat': 29.774065, 'LT_lon': 115.970635, 'RB_lat': 29.702283, 'RB_lon': 115.996851}
[SIFT] Refe

,method,site,filename,gt_lat,gt_lon,pred_lat,pred_lon,pred_x,pred_y,vote_count,error_m,matched,time_s,inliers
0,main,01,01_0514.JPG,29.746582,115.987151,29.722823,115.985777,5644.666992,19103.375000,224.0,2648.110991,True,20.884691,NaN
1,main,01,01_0564.JPG,29.710950,115.987151,29.722823,115.985777,5644.666992,19103.375000,199.0,1328.450298,True,10.151704,NaN
2,main,01,01_0742.JPG,29.719102,115.993340,29.722823,115.985777,5644.666992,19103.375000,254.0,840.416773,True,12.250175,NaN
3,main,02,02_0024.JPG,29.783184,116.040684,29.784749,116.048361,5439.726074,12164.114258,412.0,761.869078,True,19.460860,NaN
4,main,02,02_0130.JPG,29.749119,116.042747,29.784749,116.048361,5439.726074,12164.114258,458.0,4003.245228,True,10.216978,NaN
5,main,02,02_0226.JPG,29.789607,116.044832,29.784749,116.048361,5439.726074,12164.114258,346.0,639.288171,True,7.318625,NaN
6,main,03,03_0008.JPG,32.308276,119.891281,32.330963,119.888151,30654.162109,9144.165039,33.0,2542.532725,True,34.354827,NaN
7,main,03,03_0126.JPG,32.337417,119.842316,32.330963,119.888151,30654.162109,9144.165039,285.0,4370.609519,True,18.754444,NaN
8,sift,01,01_0514.JPG,29.746582,115.987151,29.717500,115.978075,206.881504,1575.231892,NaN,3354.143615,True,2.256924,29.0
9,sift,01,01_0564.JPG,29.710950,115.987151,29.713476,115.991482,579.703730,1687.284367,NaN,504.427155,True,1.687788,34.0


In [27]:
summary_df = summarize_metrics(results_df)
summary_df

,Method,Mean (m),Med. (m),RMSE (m),R@10m (%),R@50m (%),Time (s),Matched / Total
0,main,2141.815348,1935.491512,2551.111983,0.0,0.0,16.674038,8/8
1,sift,2634.502626,2385.737329,3037.126610,0.0,0.0,3.906242,8/8
2,orb,NaN,NaN,NaN,NaN,NaN,0.046683,0/8


In [28]:
raw_csv_path = OUTPUT_DIR / 'method_comparison_raw_results.csv'
summary_csv_path = OUTPUT_DIR / 'method_comparison_summary.csv'
results_df.to_csv(raw_csv_path, index=False)
summary_df.to_csv(summary_csv_path, index=False)
print('Saved raw results   ->', raw_csv_path)
print('Saved summary table ->', summary_csv_path)

Saved raw results   -> D:\bk_study_stuff\paper6\LocalizationUAV\outputs\method_comparison\method_comparison_raw_results.csv
Saved summary table -> D:\bk_study_stuff\paper6\LocalizationUAV\outputs\method_comparison\method_comparison_summary.csv
